In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
import tensorflow as tf
from typing import Tuple
import numpy as np

class TimeAnchoredPositionalEncoding(tf.keras.layers.Layer):
    """
    Injects absolute elapsed time into the epoch embeddings using sinusoidal encodings.
    Prevents sequence corruption when artifact rejection creates temporal gaps.
    """
    def __init__(self, embed_dim: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim

    def call(self, x: tf.Tensor, times: tf.Tensor) -> tf.Tensor:
        """
        Args:
            x: Tensor of shape (1, N_actual, embed_dim)
            times: Tensor of shape (1, N_actual) representing elapsed seconds
        Returns:
            Tensor of shape (1, N_actual, embed_dim)
        """
        times_expanded = tf.expand_dims(times, axis=-1)  
        
        half_dim = self.embed_dim // 2
        indices = tf.range(half_dim, dtype=tf.float32)
        div_term = tf.exp(indices * -(tf.math.log(10000.0) / half_dim))
        
        scaled_time = times_expanded * div_term  
        
        pe_sin = tf.math.sin(scaled_time)
        pe_cos = tf.math.cos(scaled_time)
        
        pe = tf.concat([pe_sin, pe_cos], axis=-1)  
        return x + pe


class IntraEpochEncoder1D(tf.keras.layers.Layer):
    """
    Multi-branch 1D CNN for raw EEG feature extraction. 
    Bypasses rigid STFT limitations and VRAM explosion.
    """
    def __init__(self, embed_dim: int = 64, **kwargs):
        super().__init__(**kwargs)
        # Branch A: Low-Frequency (Delta/Theta Tracker)
        self.conv_a = tf.keras.layers.Conv1D(16, kernel_size=64, strides=8, padding='same', use_bias=False)
        self.bn_a = tf.keras.layers.BatchNormalization()
        self.relu_a = tf.keras.layers.ReLU()
        self.pool_a = tf.keras.layers.MaxPooling1D(pool_size=4)
        
        # Branch B: High-Frequency (Spindles/EMG Tracker)
        self.conv_b = tf.keras.layers.Conv1D(16, kernel_size=16, strides=8, padding='same', use_bias=False)
        self.bn_b = tf.keras.layers.BatchNormalization()
        self.relu_b = tf.keras.layers.ReLU()
        self.pool_b = tf.keras.layers.MaxPooling1D(pool_size=4)
        
        # Fusion Head
        self.gap = tf.keras.layers.GlobalAveragePooling1D()
        self.proj = tf.keras.layers.Dense(embed_dim)
        self.ln = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    def call(self, x: tf.Tensor, training: bool = False) -> tf.Tensor:
        """
        Args:
            x: Tensor of shape (N_actual, 3840, 9)
        Returns:
            Tensor of shape (N_actual, embed_dim)
        """
        a = self.pool_a(self.relu_a(self.bn_a(self.conv_a(x), training=training)))
        b = self.pool_b(self.relu_b(self.bn_b(self.conv_b(x), training=training)))
        
        merged = tf.concat([a, b], axis=-1)
        z = self.gap(merged)
        return self.ln(self.proj(z))


class SleepArchitectureModel(tf.keras.Model):
    """
    Final Multi-Task Model with Time-Aware Bi-GRU Dynamics.
    Requires batch_size=1 to avoid padding corruption.
    """
    def __init__(self, embed_dim: int = 64, gru_units: int = 32, num_stages: int = 5, **kwargs):
        super().__init__(**kwargs)
        self.encoder = IntraEpochEncoder1D(embed_dim=embed_dim)
        self.pe_layer = TimeAnchoredPositionalEncoding(embed_dim=embed_dim)
        
        self.bi_gru = tf.keras.layers.Bidirectional(
            tf.keras.layers.GRU(gru_units, return_sequences=True)
        )
        
        # Diagnostic Head (Global Readout)
        self.attention_scorer = tf.keras.layers.Dense(1, activation=None)
        self.clf_hidden = tf.keras.layers.Dense(32, activation='relu')
        self.clf_drop = tf.keras.layers.Dropout(0.5)
        self.clf_out = tf.keras.layers.Dense(1, activation='sigmoid')
        
        # Auxiliary Stage Head (Local Readout)
        self.stage_out = tf.keras.layers.Dense(num_stages, activation='softmax')

    def call(self, inputs: Tuple[tf.Tensor, tf.Tensor], training: bool = False) -> Tuple[tf.Tensor, tf.Tensor]:
        """
        Args:
            inputs[0] -> X_eeg: (1, N_actual, 3840, 9)
            inputs[1] -> T_sec: (1, N_actual)
        Returns:
            P_insomnia: (1, 1)
            P_stages: (1, N_actual, 5)
        """
        x_eeg, t_sec = inputs
        n_actual = tf.shape(x_eeg)[1]
        
        # 1. Intra-Epoch Feature Extraction
        x_reshaped = tf.reshape(x_eeg, (n_actual, 3840, 9))
        z = self.encoder(x_reshaped, training=training)
        z = tf.reshape(z, (1, n_actual, -1)) 
        
        # 2. Time-Anchored Positional Encoding
        z_pos = self.pe_layer(z, t_sec)
        
        # 3. Explicit Time-Aware Delta Injection
        # Shift times by 1 to compute elapsed time between epochs
        t_sec_shifted = tf.pad(t_sec[:, :-1], [[0, 0], [1, 0]], constant_values=t_sec[0, 0] - 30.0)
        delta_t = t_sec - t_sec_shifted  
        
        # Enforce strictly positive delta_t to prevent NaN from log1p
        delta_t_safe = tf.math.maximum(delta_t, 0.0)
        delta_t_norm = tf.math.log1p(delta_t_safe) 
        delta_t_norm = tf.expand_dims(delta_t_norm, axis=-1) 
        
        # Append delta_t as the 65th feature dimension
        z_aware = tf.concat([z_pos, delta_t_norm], axis=-1) 
        
        # 4. Sequence Processing
        h = self.bi_gru(z_aware, training=training) 
        
        # 5a. Diagnostic Head (Insomnia Prediction)
        attn_logits = self.attention_scorer(h) 
        attn_weights = tf.nn.softmax(attn_logits, axis=1)
        global_repr = tf.reduce_sum(h * attn_weights, axis=1) 
        
        p_insomnia = self.clf_out(self.clf_drop(self.clf_hidden(global_repr), training=training))
        
        # 5b. Auxiliary Head (Stage Prediction)
        p_stages = self.stage_out(h) 
        
        return p_insomnia, p_stages


# ==========================================
# MINIMAL REPRODUCIBLE EXAMPLE (SHAPE VERIFICATION)
# ==========================================
if __name__ == "__main__":
    # Simulate a dynamic subject night (e.g., 619 epochs for Insomnia_04)
    N_EPOCHS = 619
    dummy_eeg = tf.random.normal((1, N_EPOCHS, 3840, 9))
    
    # Simulate times with an arbitrary 5-minute artifact gap injected at epoch 300
    base_times = np.arange(0, N_EPOCHS * 30, 30, dtype=np.float32)
    base_times[300:] += 300.0  # Inject the gap
    dummy_times = tf.convert_to_tensor(base_times.reshape(1, N_EPOCHS))

    # Initialize model
    model = SleepArchitectureModel()
    
    # Forward Pass
    p_insomnia, p_stages = model((dummy_eeg, dummy_times), training=False)
    
    # Assertions
    assert p_insomnia.shape == (1, 1), f"Expected Insomnia Output (1, 1), got {p_insomnia.shape}"
    assert p_stages.shape == (1, N_EPOCHS, 5), f"Expected Stage Output (1, {N_EPOCHS}, 5), got {p_stages.shape}"
    
    print("SUCCESS: Architectural Forward Pass Passed.")
    print(f"X_eeg Tensor:      {dummy_eeg.shape}")
    print(f"T_sec Tensor:      {dummy_times.shape}")
    print(f"P_insomnia Output: {p_insomnia.shape}")
    print(f"P_stages Output:   {p_stages.shape}")

SUCCESS: Architectural Forward Pass Passed.
X_eeg Tensor:      (1, 619, 3840, 9)
T_sec Tensor:      (1, 619)
P_insomnia Output: (1, 1)
P_stages Output:   (1, 619, 5)


In [1]:
import os
import glob
import numpy as np
import tensorflow as tf
from typing import Tuple, List, Dict

# ======================================================================
# CONFIGURATION & HYPERPARAMETERS
# ======================================================================
DATA_DIR = "/kaggle/input/datasets/johanliebert28/insominia-sliced-chronological-dataset"  # Path to your .npy files
EPOCHS = 20
ACCUMULATION_STEPS = 4
LAMBDA_AUX = 0.5  # Weight of the stage-prediction loss
LEARNING_RATE = 1e-4

# ======================================================================
# MODULE 1: DATA PIPELINE (RAM Caching & Preprocessing)
# ======================================================================
def load_and_cache_dataset(data_dir: str) -> Dict[str, Tuple[tf.Tensor, tf.Tensor, tf.Tensor, tf.Tensor]]:
    """
    Loads all subjects into RAM. 
    Applies on-the-fly decimation (256Hz -> 128Hz) and shape transpositions.
    """
    subject_cache = {}
    data_files = glob.glob(os.path.join(data_dir, "*_data.npy"))
    
    for df in data_files:
        subject_id = os.path.basename(df).replace("_data.npy", "")
        
        # Determine Binary Label (Insomnia=1, Normal=0)
        y_insom_val = 1.0 if "Insomnia" in subject_id else 0.0
        
        # Load raw numpy arrays
        x_raw = np.load(df)  # Shape expected: (N_actual, 9, 7680)
        y_stages_raw = np.load(os.path.join(data_dir, f"{subject_id}_labels.npy")) # (N_actual,)
        t_sec_raw = np.load(os.path.join(data_dir, f"{subject_id}_times.npy"))     # (N_actual,)
        
        # CRITICAL FIX 1: Downsample to 128Hz to prevent VRAM explosion
        x_downsampled = x_raw[:, :, ::2]  # Takes every 2nd element -> (N_actual, 9, 3840)
        
        # CRITICAL FIX 2: Transpose for 1D CNN (Channels Last)
        x_transposed = np.transpose(x_downsampled, (0, 2, 1)) # -> (N_actual, 3840, 9)
        
        # Convert to TensorFlow tensors and add strictly Batch=1 dimension
        x_tensor = tf.convert_to_tensor(np.expand_dims(x_transposed, axis=0), dtype=tf.float32)
        t_tensor = tf.convert_to_tensor(np.expand_dims(t_sec_raw, axis=0), dtype=tf.float32)
        y_insom_tensor = tf.convert_to_tensor([[y_insom_val]], dtype=tf.float32)
        y_stages_tensor = tf.convert_to_tensor(np.expand_dims(y_stages_raw, axis=0), dtype=tf.int32)
        
        subject_cache[subject_id] = (x_tensor, t_tensor, y_insom_tensor, y_stages_tensor)
        print(f"Loaded {subject_id}: X={x_tensor.shape} | T={t_tensor.shape}")
        
    return subject_cache

# ======================================================================
# MODULE 2: NEURAL NETWORK ARCHITECTURE
# ======================================================================
class TimeAnchoredPositionalEncoding(tf.keras.layers.Layer):
    """STAGE 2.1: Absolute Time Injection"""
    def __init__(self, embed_dim: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim

    def call(self, x: tf.Tensor, times: tf.Tensor) -> tf.Tensor:
        times_expanded = tf.expand_dims(times, axis=-1)  
        half_dim = self.embed_dim // 2
        indices = tf.range(half_dim, dtype=tf.float32)
        div_term = tf.exp(indices * -(tf.math.log(10000.0) / half_dim))
        scaled_time = times_expanded * div_term  
        pe = tf.concat([tf.math.sin(scaled_time), tf.math.cos(scaled_time)], axis=-1)  
        return x + pe

class IntraEpochEncoder1D(tf.keras.layers.Layer):
    """STAGE 1: 1D Multi-Branch CNN (Intra-Epoch)"""
    def __init__(self, embed_dim: int = 64, **kwargs):
        super().__init__(**kwargs)
        # Branch A: Delta/Theta
        self.conv_a = tf.keras.layers.Conv1D(16, 64, strides=8, padding='same', use_bias=False)
        self.bn_a = tf.keras.layers.BatchNormalization()
        self.relu_a = tf.keras.layers.ReLU()
        self.pool_a = tf.keras.layers.MaxPooling1D(4)
        
        # Branch B: Spindles/EMG
        self.conv_b = tf.keras.layers.Conv1D(16, 16, strides=8, padding='same', use_bias=False)
        self.bn_b = tf.keras.layers.BatchNormalization()
        self.relu_b = tf.keras.layers.ReLU()
        self.pool_b = tf.keras.layers.MaxPooling1D(4)
        
        self.gap = tf.keras.layers.GlobalAveragePooling1D()
        self.proj = tf.keras.layers.Dense(embed_dim)
        self.ln = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    def call(self, x: tf.Tensor, training: bool = False) -> tf.Tensor:
        a = self.pool_a(self.relu_a(self.bn_a(self.conv_a(x), training=training)))
        b = self.pool_b(self.relu_b(self.bn_b(self.conv_b(x), training=training)))
        return self.ln(self.proj(self.gap(tf.concat([a, b], axis=-1))))

class SleepArchitectureModel(tf.keras.Model):
    """FULL MODEL ASSEMBLY"""
    def __init__(self, embed_dim: int = 64, gru_units: int = 32, num_stages: int = 5, **kwargs):
        super().__init__(**kwargs)
        self.encoder = IntraEpochEncoder1D(embed_dim=embed_dim)
        self.pe_layer = TimeAnchoredPositionalEncoding(embed_dim=embed_dim)
        
        # STAGE 2.2: Time-Aware Sequence Processing
        self.bi_gru = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(gru_units, return_sequences=True))
        
        # STAGE 3.1: Diagnostic Head (Global Attention)
        self.attention_scorer = tf.keras.layers.Dense(1, activation=None)
        self.clf_hidden = tf.keras.layers.Dense(32, activation='relu')
        self.clf_drop = tf.keras.layers.Dropout(0.5)
        self.clf_out = tf.keras.layers.Dense(1, activation='sigmoid')
        
        # STAGE 3.2: Auxiliary Stage Head
        self.stage_out = tf.keras.layers.Dense(num_stages, activation='softmax')

    def call(self, inputs: Tuple[tf.Tensor, tf.Tensor], training: bool = False) -> Tuple[tf.Tensor, tf.Tensor]:
        x_eeg, t_sec = inputs
        n_actual = tf.shape(x_eeg)[1]
        
        # STAGE 1 Process
        x_reshaped = tf.reshape(x_eeg, (n_actual, 3840, 9))
        z = self.encoder(x_reshaped, training=training)
        z = tf.reshape(z, (1, n_actual, -1)) 
        
        # STAGE 2 Process (Position & Delta T)
        z_pos = self.pe_layer(z, t_sec)
        t_sec_shifted = tf.pad(t_sec[:, :-1], [[0, 0], [1, 0]], constant_values=t_sec[0, 0] - 30.0)
        delta_t_safe = tf.math.maximum(t_sec - t_sec_shifted, 0.0)
        delta_t_norm = tf.expand_dims(tf.math.log1p(delta_t_safe), axis=-1) 
        
        z_aware = tf.concat([z_pos, delta_t_norm], axis=-1) 
        h = self.bi_gru(z_aware, training=training) 
        
        # STAGE 3 Process (Bifurcated Outputs)
        attn_weights = tf.nn.softmax(self.attention_scorer(h), axis=1)
        global_repr = tf.reduce_sum(h * attn_weights, axis=1) 
        
        p_insomnia = self.clf_out(self.clf_drop(self.clf_hidden(global_repr), training=training))
        p_stages = self.stage_out(h) 
        
        return p_insomnia, p_stages

# ======================================================================
# MODULE 3: GRADIENT ACCUMULATION ENGINE
# ======================================================================
bce_loss_fn = tf.keras.losses.BinaryCrossentropy(reduction=tf.keras.losses.Reduction.SUM)
scce_loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(reduction=tf.keras.losses.Reduction.SUM)

@tf.function
def accumulate_step(x_eeg, t_sec, y_insom, y_stages, model, accumulated_grads, lambda_aux):
    """Executes forward pass and accumulates gradients without applying them."""
    with tf.GradientTape() as tape:
        p_insom, p_stages = model((x_eeg, t_sec), training=True)
        
        loss_bce = bce_loss_fn(y_insom, p_insom)
        n_actual = tf.cast(tf.shape(y_stages)[1], tf.float32)
        loss_scce = scce_loss_fn(y_stages, p_stages) / n_actual
        
        total_loss = loss_bce + (lambda_aux * loss_scce)

    gradients = tape.gradient(total_loss, model.trainable_variables)
    for i, grad in enumerate(gradients):
        if grad is not None:
            accumulated_grads[i].assign_add(grad)
            
    return total_loss, loss_bce, loss_scce

# ======================================================================
# MODULE 4: LEAVE-ONE-SUBJECT-OUT (LOSO) VALIDATION
# ======================================================================
def run_loso_validation(subject_cache: dict):
    subject_ids = list(subject_cache.keys())
    
    for fold_idx, test_subject in enumerate(subject_ids):
        print(f"\n{'='*60}")
        print(f"FOLD {fold_idx+1}/{len(subject_ids)} — Testing on: {test_subject}")
        print(f"{'='*60}")
        
        # Isolate Training Subjects
        train_subjects = [s for s in subject_ids if s != test_subject]
        
        # Initialize Fresh Model & Optimizer for this Fold
        model = SleepArchitectureModel()
        optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
        
        # Initialize Gradient Accumulators
        # (We trigger a dummy forward pass to build the model's weights first)
        dummy_x, dummy_t, _, _ = subject_cache[train_subjects[0]]
        model((dummy_x, dummy_t), training=False) 
        
        accumulated_grads = [tf.Variable(tf.zeros_like(var), trainable=False) for var in model.trainable_variables]
        
        # Training Loop
        for epoch in range(EPOCHS):
            epoch_bce, epoch_scce = 0.0, 0.0
            np.random.shuffle(train_subjects) # Shuffle training order
            
            for step, sub_id in enumerate(train_subjects):
                x_eeg, t_sec, y_insom, y_stages = subject_cache[sub_id]
                
                t_loss, b_loss, s_loss = accumulate_step(
                    x_eeg, t_sec, y_insom, y_stages, model, accumulated_grads, tf.constant(LAMBDA_AUX)
                )
                
                epoch_bce += b_loss.numpy()
                epoch_scce += s_loss.numpy()
                
                # Apply Gradients every N steps
                if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_subjects):
                    averaged_grads = [g / tf.cast(ACCUMULATION_STEPS, tf.float32) for g in accumulated_grads]
                    optimizer.apply_gradients(zip(averaged_grads, model.trainable_variables))
                    for var in accumulated_grads:
                        var.assign(tf.zeros_like(var))
                        
            print(f"  Epoch {epoch+1:02d}/{EPOCHS} | Train BCE: {epoch_bce/len(train_subjects):.4f} | Aux SCCE: {epoch_scce/len(train_subjects):.4f}")
            
        # Evaluation Phase (Inference on Left-Out Subject)
        test_x, test_t, test_y_insom, _ = subject_cache[test_subject]
        p_insom_test, _ = model((test_x, test_t), training=False)
        
        pred_prob = p_insom_test.numpy()[0][0]
        true_label = test_y_insom.numpy()[0][0]
        print(f"  [EVAL] {test_subject} | P(Insomnia) = {pred_prob:.4f} | True = {true_label}")
        
if __name__ == "__main__":
    if not os.path.exists(DATA_DIR):
        print(f"FATAL: {DATA_DIR} not found. Please verify your file paths.")
    else:
        # Load all subjects into RAM 
        print("Initializing Caching Protocol...")
        cached_data = load_and_cache_dataset(DATA_DIR)
        
        if len(cached_data) > 0:
            # Start strict LOSO Validation
            run_loso_validation(cached_data)
        else:
            print("No valid .npy subjects found in directory.")

2026-03-09 21:25:37.816431: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773091538.030600      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773091538.091061      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773091538.613439      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773091538.613477      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773091538.613481      55 computation_placer.cc:177] computation placer alr

Initializing Caching Protocol...


I0000 00:00:1773091566.656309      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773091566.662280      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Loaded Normal_03: X=(1, 614, 3840, 9) | T=(1, 614)
Loaded Insomnia_09: X=(1, 390, 3840, 9) | T=(1, 390)
Loaded Insomnia_08: X=(1, 338, 3840, 9) | T=(1, 338)
Loaded Insomnia_05: X=(1, 567, 3840, 9) | T=(1, 567)
Loaded Normal_05: X=(1, 574, 3840, 9) | T=(1, 574)
Loaded Normal_07: X=(1, 772, 3840, 9) | T=(1, 772)
Loaded Insomnia_02: X=(1, 748, 3840, 9) | T=(1, 748)
Loaded Normal_10: X=(1, 631, 3840, 9) | T=(1, 631)
Loaded Insomnia_10: X=(1, 647, 3840, 9) | T=(1, 647)
Loaded Normal_09: X=(1, 596, 3840, 9) | T=(1, 596)
Loaded Normal_01: X=(1, 660, 3840, 9) | T=(1, 660)
Loaded Insomnia_03: X=(1, 677, 3840, 9) | T=(1, 677)
Loaded Insomnia_01: X=(1, 780, 3840, 9) | T=(1, 780)
Loaded Normal_06: X=(1, 553, 3840, 9) | T=(1, 553)
Loaded Normal_08: X=(1, 564, 3840, 9) | T=(1, 564)
Loaded Normal_04: X=(1, 376, 3840, 9) | T=(1, 376)
Loaded Normal_02: X=(1, 676, 3840, 9) | T=(1, 676)
Loaded Insomnia_06: X=(1, 669, 3840, 9) | T=(1, 669)
Loaded Normal_11: X=(1, 689, 3840, 9) | T=(1, 689)
Loaded Insomnia

I0000 00:00:1773091597.396286      55 cuda_dnn.cc:529] Loaded cuDNN version 91002


  Epoch 01/20 | Train BCE: 0.7101 | Aux SCCE: 1.8175
  Epoch 02/20 | Train BCE: 0.7730 | Aux SCCE: 1.6551
  Epoch 03/20 | Train BCE: 0.6988 | Aux SCCE: 1.6634
  Epoch 04/20 | Train BCE: 0.6661 | Aux SCCE: 1.6584
  Epoch 05/20 | Train BCE: 0.8380 | Aux SCCE: 1.6434
  Epoch 06/20 | Train BCE: 0.6901 | Aux SCCE: 1.6533
  Epoch 07/20 | Train BCE: 0.6902 | Aux SCCE: 1.6397
  Epoch 08/20 | Train BCE: 0.6976 | Aux SCCE: 1.6198
  Epoch 09/20 | Train BCE: 0.7062 | Aux SCCE: 1.6107
  Epoch 10/20 | Train BCE: 0.7120 | Aux SCCE: 1.6014
  Epoch 11/20 | Train BCE: 0.6798 | Aux SCCE: 1.5946
  Epoch 12/20 | Train BCE: 0.7128 | Aux SCCE: 1.5910
  Epoch 13/20 | Train BCE: 0.7115 | Aux SCCE: 1.5921
  Epoch 14/20 | Train BCE: 0.6917 | Aux SCCE: 1.5871
  Epoch 15/20 | Train BCE: 0.6860 | Aux SCCE: 1.5842
  Epoch 16/20 | Train BCE: 0.6950 | Aux SCCE: 1.5780
  Epoch 17/20 | Train BCE: 0.7124 | Aux SCCE: 1.5725
  Epoch 18/20 | Train BCE: 0.7175 | Aux SCCE: 1.5740
  Epoch 19/20 | Train BCE: 0.7063 | Aux SCCE: 

In [1]:
import os
import glob
import time
import numpy as np
import tensorflow as tf
from typing import Tuple, List, Dict
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

# ======================================================================
# CONFIGURATION & HYPERPARAMETERS
# ======================================================================
DATA_DIR = "/kaggle/input/datasets/johanliebert28/insominia-sliced-chronological-dataset"  # Path to your .npy files
EPOCHS = 100                      # [FIX 3] Extended training duration
ACCUMULATION_STEPS = 4
LAMBDA_AUX = 0.1                  # [FIX 1] Reduced to prevent gradient hijacking
INITIAL_LR = 5e-4                 # [FIX 3] Higher starting LR for Cosine Decay

# ======================================================================
# MODULE 1: DATA PIPELINE (RAM Caching & Preprocessing)
# ======================================================================
def load_and_cache_dataset(data_dir: str) -> Dict[str, Tuple[tf.Tensor, tf.Tensor, tf.Tensor, tf.Tensor]]:
    subject_cache = {}
    data_files = glob.glob(os.path.join(data_dir, "*_data.npy"))
    
    for df in data_files:
        subject_id = os.path.basename(df).replace("_data.npy", "")
        y_insom_val = 1.0 if "Insomnia" in subject_id else 0.0
        
        x_raw = np.load(df)  
        y_stages_raw = np.load(os.path.join(data_dir, f"{subject_id}_labels.npy")) 
        t_sec_raw = np.load(os.path.join(data_dir, f"{subject_id}_times.npy"))     
        
        # 256Hz -> 128Hz Downsampling
        x_downsampled = x_raw[:, :, ::2]  
        # Transpose to (N_actual, 3840, 9)
        x_transposed = np.transpose(x_downsampled, (0, 2, 1)) 
        
        x_tensor = tf.convert_to_tensor(np.expand_dims(x_transposed, axis=0), dtype=tf.float32)
        t_tensor = tf.convert_to_tensor(np.expand_dims(t_sec_raw, axis=0), dtype=tf.float32)
        y_insom_tensor = tf.convert_to_tensor([[y_insom_val]], dtype=tf.float32)
        y_stages_tensor = tf.convert_to_tensor(np.expand_dims(y_stages_raw, axis=0), dtype=tf.int32)
        
        subject_cache[subject_id] = (x_tensor, t_tensor, y_insom_tensor, y_stages_tensor)
        print(f"Loaded {subject_id}: X={x_tensor.shape} | T={t_tensor.shape}")
        
    return subject_cache

# ======================================================================
# MODULE 2: NEURAL NETWORK ARCHITECTURE
# ======================================================================
class TimeAnchoredPositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, embed_dim: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim

    def call(self, x: tf.Tensor, times: tf.Tensor) -> tf.Tensor:
        times_expanded = tf.expand_dims(times, axis=-1)  
        half_dim = self.embed_dim // 2
        indices = tf.range(half_dim, dtype=tf.float32)
        div_term = tf.exp(indices * -(tf.math.log(10000.0) / half_dim))
        scaled_time = times_expanded * div_term  
        pe = tf.concat([tf.math.sin(scaled_time), tf.math.cos(scaled_time)], axis=-1)  
        return x + pe

class IntraEpochEncoder1D(tf.keras.layers.Layer):
    def __init__(self, embed_dim: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.conv_a = tf.keras.layers.Conv1D(16, 64, strides=8, padding='same', use_bias=False)
        self.bn_a = tf.keras.layers.BatchNormalization()
        self.relu_a = tf.keras.layers.ReLU()
        self.pool_a = tf.keras.layers.MaxPooling1D(4)
        
        self.conv_b = tf.keras.layers.Conv1D(16, 16, strides=8, padding='same', use_bias=False)
        self.bn_b = tf.keras.layers.BatchNormalization()
        self.relu_b = tf.keras.layers.ReLU()
        self.pool_b = tf.keras.layers.MaxPooling1D(4)
        
        self.gap = tf.keras.layers.GlobalAveragePooling1D()
        self.proj = tf.keras.layers.Dense(embed_dim)
        self.ln = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    def call(self, x: tf.Tensor, training: bool = False) -> tf.Tensor:
        a = self.pool_a(self.relu_a(self.bn_a(self.conv_a(x), training=training)))
        b = self.pool_b(self.relu_b(self.bn_b(self.conv_b(x), training=training)))
        return self.ln(self.proj(self.gap(tf.concat([a, b], axis=-1))))

class SleepArchitectureModel(tf.keras.Model):
    def __init__(self, embed_dim: int = 64, gru_units: int = 32, num_stages: int = 5, **kwargs):
        super().__init__(**kwargs)
        self.encoder = IntraEpochEncoder1D(embed_dim=embed_dim)
        self.pe_layer = TimeAnchoredPositionalEncoding(embed_dim=embed_dim)
        self.bi_gru = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(gru_units, return_sequences=True))
        
        self.attention_scorer = tf.keras.layers.Dense(1, activation=None)
        self.clf_hidden = tf.keras.layers.Dense(32, activation='relu')
        self.clf_drop = tf.keras.layers.Dropout(0.5)
        self.clf_out = tf.keras.layers.Dense(1, activation='sigmoid')
        self.stage_out = tf.keras.layers.Dense(num_stages, activation='softmax')

    def call(self, inputs: Tuple[tf.Tensor, tf.Tensor], training: bool = False) -> Tuple[tf.Tensor, tf.Tensor]:
        x_eeg, t_sec = inputs
        n_actual = tf.shape(x_eeg)[1]
        
        x_reshaped = tf.reshape(x_eeg, (n_actual, 3840, 9))
        z = self.encoder(x_reshaped, training=training)
        z = tf.reshape(z, (1, n_actual, -1)) 
        
        z_pos = self.pe_layer(z, t_sec)
        t_sec_shifted = tf.pad(t_sec[:, :-1], [[0, 0], [1, 0]], constant_values=t_sec[0, 0] - 30.0)
        delta_t_safe = tf.math.maximum(t_sec - t_sec_shifted, 0.0)
        delta_t_norm = tf.expand_dims(tf.math.log1p(delta_t_safe), axis=-1) 
        
        z_aware = tf.concat([z_pos, delta_t_norm], axis=-1) 
        h = self.bi_gru(z_aware, training=training) 
        
        attn_weights = tf.nn.softmax(self.attention_scorer(h), axis=1)
        global_repr = tf.reduce_sum(h * attn_weights, axis=1) 
        
        p_insomnia = self.clf_out(self.clf_drop(self.clf_hidden(global_repr), training=training))
        p_stages = self.stage_out(h) 
        
        return p_insomnia, p_stages

# ======================================================================
# MODULE 3: GRADIENT ACCUMULATION & LOSS (With Fixes)
# ======================================================================
# [FIX 4] Added label smoothing to diagnostic head
bce_loss_fn = tf.keras.losses.BinaryCrossentropy(
    reduction=tf.keras.losses.Reduction.SUM, 
    label_smoothing=0.1
)
scce_loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    reduction=tf.keras.losses.Reduction.SUM
)

@tf.function
def accumulate_step(x_eeg, t_sec, y_insom, y_stages, model, accumulated_grads, lambda_aux):
    with tf.GradientTape() as tape:
        p_insom, p_stages = model((x_eeg, t_sec), training=True)
        loss_bce = bce_loss_fn(y_insom, p_insom)
        
        # [FIX 2] Dynamic Class Weighting (Inverse Frequency)
        y_stages_flat = tf.reshape(y_stages, [-1])
        n_actual = tf.cast(tf.shape(y_stages_flat)[0], tf.float32)
        
        counts = tf.math.bincount(tf.cast(y_stages_flat, tf.int32), minlength=5, maxlength=5, dtype=tf.float32)
        # Bounding max(counts, 1.0) prevents div-by-zero if a stage is missing
        class_weights = n_actual / (5.0 * tf.math.maximum(counts, 1.0))
        
        sample_weights = tf.gather(class_weights, tf.cast(y_stages_flat, tf.int32))
        sample_weights = tf.reshape(sample_weights, tf.shape(y_stages))
        
        # Apply sample weights to the stage loss
        loss_scce = scce_loss_fn(y_stages, p_stages, sample_weight=sample_weights) / n_actual
        
        total_loss = loss_bce + (lambda_aux * loss_scce)

    gradients = tape.gradient(total_loss, model.trainable_variables)
    for i, grad in enumerate(gradients):
        if grad is not None:
            accumulated_grads[i].assign_add(grad)
            
    return total_loss, loss_bce, loss_scce

# ======================================================================
# MODULE 4: LEAVE-ONE-SUBJECT-OUT (LOSO) VALIDATION
# ======================================================================
def run_loso_validation(subject_cache: dict):
    subject_ids = list(subject_cache.keys())
    
    # Trackers for Publication Metrics
    all_y_true = []
    all_y_pred_probs = []
    inference_latencies = []
    total_params = 0
    
    for fold_idx, test_subject in enumerate(subject_ids):
        print(f"\n{'='*60}")
        print(f"FOLD {fold_idx+1}/{len(subject_ids)} — Testing on: {test_subject}")
        print(f"{'='*60}")
        
        train_subjects = [s for s in subject_ids if s != test_subject]
        
        model = SleepArchitectureModel()
        
        # [FIX 3] Cosine Decay Restarts Schedule
        steps_per_epoch = len(train_subjects)
        lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
            initial_learning_rate=INITIAL_LR,
            first_decay_steps=steps_per_epoch * 20, 
            t_mul=2.0,
            m_mul=0.9,
            alpha=1e-5
        )
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)
        
        # Dummy pass to initialize weights
        dummy_x, dummy_t, _, _ = subject_cache[train_subjects[0]]
        model((dummy_x, dummy_t), training=False) 
        if fold_idx == 0:
            total_params = model.count_params()
        
        accumulated_grads = [tf.Variable(tf.zeros_like(var), trainable=False) for var in model.trainable_variables]
        
        # Training Loop
        for epoch in range(EPOCHS):
            epoch_bce, epoch_scce = 0.0, 0.0
            np.random.shuffle(train_subjects) 
            
            for step, sub_id in enumerate(train_subjects):
                x_eeg, t_sec, y_insom, y_stages = subject_cache[sub_id]
                
                t_loss, b_loss, s_loss = accumulate_step(
                    x_eeg, t_sec, y_insom, y_stages, model, accumulated_grads, tf.constant(LAMBDA_AUX)
                )
                
                epoch_bce += b_loss.numpy()
                epoch_scce += s_loss.numpy()
                
                if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_subjects):
                    averaged_grads = [g / tf.cast(ACCUMULATION_STEPS, tf.float32) for g in accumulated_grads]
                    optimizer.apply_gradients(zip(averaged_grads, model.trainable_variables))
                    for var in accumulated_grads:
                        var.assign(tf.zeros_like(var))
                        
            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(f"  Epoch {epoch+1:03d}/{EPOCHS} | Train BCE: {epoch_bce/len(train_subjects):.4f} | Aux SCCE: {epoch_scce/len(train_subjects):.4f} | LR: {optimizer.learning_rate.numpy():.6f}")
            
        # Evaluation Phase (Inference Profiling)
        test_x, test_t, test_y_insom, _ = subject_cache[test_subject]
        
        # Measure Latency
        start_time = time.perf_counter()
        p_insom_test, _ = model((test_x, test_t), training=False)
        inf_latency_ms = (time.perf_counter() - start_time) * 1000
        
        pred_prob = p_insom_test.numpy()[0][0]
        true_label = test_y_insom.numpy()[0][0]
        
        all_y_pred_probs.append(pred_prob)
        all_y_true.append(true_label)
        inference_latencies.append(inf_latency_ms)
        
        print(f"  [EVAL] {test_subject} | P(Insomnia) = {pred_prob:.4f} | True = {true_label} | Latency: {inf_latency_ms:.1f} ms")
        
    # ======================================================================
    # FINAL PUBLICATION METRICS (Aggregated across all LOSO folds)
    # ======================================================================
    print(f"\n{'='*60}")
    print("FINAL LOSO VALIDATION METRICS (PUBLICATION READY)")
    print(f"{'='*60}")
    
    y_true_arr = np.array(all_y_true)
    y_prob_arr = np.array(all_y_pred_probs)
    y_pred_arr = (y_prob_arr >= 0.5).astype(int)
    
    acc = accuracy_score(y_true_arr, y_pred_arr)
    f1 = f1_score(y_true_arr, y_pred_arr)
    auroc = roc_auc_score(y_true_arr, y_prob_arr)
    
    tn, fp, fn, tp = confusion_matrix(y_true_arr, y_pred_arr).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    print(f"Global Accuracy:  {acc:.4f}")
    print(f"F1-Score:         {f1:.4f}")
    print(f"AUROC:            {auroc:.4f}")
    print(f"Sensitivity:      {sensitivity:.4f}")
    print(f"Specificity:      {specificity:.4f}")
    print(f"-"*60)
    print(f"Deployment Viability Profile:")
    print(f"Parameter Count:  {total_params:,}")
    print(f"Avg Latency/Subj: {np.mean(inference_latencies):.2f} ms (Standard Deviation: {np.std(inference_latencies):.2f} ms)")
    print(f"{'='*60}\n")

if __name__ == "__main__":
    if not os.path.exists(DATA_DIR):
        print(f"FATAL: {DATA_DIR} not found. Please verify your file paths.")
    else:
        print("Initializing Caching Protocol...")
        cached_data = load_and_cache_dataset(DATA_DIR)
        
        if len(cached_data) > 0:
            run_loso_validation(cached_data)
        else:
            print("No valid .npy subjects found in directory.")

2026-03-09 21:57:20.739169: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773093440.920093      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773093440.968549      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773093441.387157      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773093441.387202      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773093441.387205      55 computation_placer.cc:177] computation placer alr

Initializing Caching Protocol...


I0000 00:00:1773093466.327348      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773093466.333514      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Loaded Normal_03: X=(1, 614, 3840, 9) | T=(1, 614)
Loaded Insomnia_09: X=(1, 390, 3840, 9) | T=(1, 390)
Loaded Insomnia_08: X=(1, 338, 3840, 9) | T=(1, 338)
Loaded Insomnia_05: X=(1, 567, 3840, 9) | T=(1, 567)
Loaded Normal_05: X=(1, 574, 3840, 9) | T=(1, 574)
Loaded Normal_07: X=(1, 772, 3840, 9) | T=(1, 772)
Loaded Insomnia_02: X=(1, 748, 3840, 9) | T=(1, 748)
Loaded Normal_10: X=(1, 631, 3840, 9) | T=(1, 631)
Loaded Insomnia_10: X=(1, 647, 3840, 9) | T=(1, 647)
Loaded Normal_09: X=(1, 596, 3840, 9) | T=(1, 596)
Loaded Normal_01: X=(1, 660, 3840, 9) | T=(1, 660)
Loaded Insomnia_03: X=(1, 677, 3840, 9) | T=(1, 677)
Loaded Insomnia_01: X=(1, 780, 3840, 9) | T=(1, 780)
Loaded Normal_06: X=(1, 553, 3840, 9) | T=(1, 553)
Loaded Normal_08: X=(1, 564, 3840, 9) | T=(1, 564)
Loaded Normal_04: X=(1, 376, 3840, 9) | T=(1, 376)
Loaded Normal_02: X=(1, 676, 3840, 9) | T=(1, 676)
Loaded Insomnia_06: X=(1, 669, 3840, 9) | T=(1, 669)
Loaded Normal_11: X=(1, 689, 3840, 9) | T=(1, 689)
Loaded Insomnia

I0000 00:00:1773093496.365535      55 cuda_dnn.cc:529] Loaded cuDNN version 91002


  Epoch 001/100 | Train BCE: 0.9076 | Aux SCCE: 1.7201 | LR: 0.000500
  Epoch 010/100 | Train BCE: 0.7344 | Aux SCCE: 1.6169 | LR: 0.000475
  Epoch 020/100 | Train BCE: 0.7045 | Aux SCCE: 1.5852 | LR: 0.000406
  Epoch 030/100 | Train BCE: 0.6786 | Aux SCCE: 1.5629 | LR: 0.000306
  Epoch 040/100 | Train BCE: 0.6813 | Aux SCCE: 1.5515 | LR: 0.000194
  Epoch 050/100 | Train BCE: 0.6805 | Aux SCCE: 1.5453 | LR: 0.000094
  Epoch 060/100 | Train BCE: 0.7140 | Aux SCCE: 1.5423 | LR: 0.000025
  Epoch 070/100 | Train BCE: 0.6930 | Aux SCCE: 1.5422 | LR: 0.000450
  Epoch 080/100 | Train BCE: 0.6881 | Aux SCCE: 1.5323 | LR: 0.000444
  Epoch 090/100 | Train BCE: 0.6759 | Aux SCCE: 1.5204 | LR: 0.000428
  Epoch 100/100 | Train BCE: 0.6826 | Aux SCCE: 1.5086 | LR: 0.000401
  [EVAL] Normal_03 | P(Insomnia) = 0.5301 | True = 0.0 | Latency: 184.8 ms

FOLD 2/22 — Testing on: Insomnia_09
  Epoch 001/100 | Train BCE: 0.8070 | Aux SCCE: 1.7774 | LR: 0.000500
  Epoch 010/100 | Train BCE: 0.7017 | Aux SCCE: 